<a href="https://colab.research.google.com/github/korzhimanov/dsp-seminars/blob/main/notes/5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Практическое занятие №5: Основные характеристики цифровых фильтров

## 1. Основные характеристики фильтров

### Амплитудно-частотная характеристика (АЧХ)
- В Python: `w, H = signal.freqz(b, a, fs=fs)`
- `H` – комплексная величина, АЧХ – её модуль `np.abs(H)`

### Фазо-частотная характеристика (ФЧХ)
- Вычисляется как `np.angle(H)`

### Групповая задержка
- $ \tau_g(f) = -\frac{d\phi(f)}{d\omega} $
- В Python: `w_gd, gd = signal.group_delay((b, a), fs=fs)`

### Импульсная характеристика
- Реакция на единичный импульс $ \delta[n] $
- `impulse = np.zeros(20); impulse[0] = 1`
- Вычисляется как `h = signal.lfilter(b, a, impulse)`

### Переходная характеристика
- Реакция на единичный скачок $ u[n] $
- `step = np.ones(50)`
- `h = signal.lfilter(b, a, step)`

## 2. Примеры задания фильтров по коэффициентам

### Скользящее среднее (КИХ)

```python
M = 5
b = np.ones(M) / M   # коэффициенты
a = [1]              # знаменатель

# Импульсная характеристика
impulse = np.zeros(20)
impulse[0] = 1
h = signal.lfilter(b, a, impulse)
```

### Экспоненциальное сглаживание (БИХ)

```python
alpha = 0.2
b = [alpha]
a = [1, -(1-alpha)]

y = signal.lfilter(b, a, x)
```

Передаточная функция $H(z) = \dfrac{\alpha}{1 - (1-\alpha)z^{-1}}$ имеет единственный полюс $z = 1 - \alpha$. Чем ближе полюс к единичной окружности (малые α), тем медленнее затухает импульсная характеристика $h[n] = \alpha(1-\alpha)^n$ и тем сильнее сглаживание. Полюса и нули находит `z, p, k = signal.tf2zpk(b, a)`.

## 3. Фильтр Савицкого – Голея (Savitzky–Golay)

Это КИХ-фильтр, основанный на локальной полиномиальной регрессии. Он сохраняет форму сигнала (пики, перегибы) лучше, чем скользящее среднее.

```python
from scipy.signal import savgol_filter
y = savgol_filter(x, window_length=11, polyorder=2)
```

Имеется два основных параметра:
- `window_length` – длина окна усреднения (порядок фильтра)
- `polyorder` – степень используемого полинома
- Увеличение `window_length` → сильнее сглаживание, но больше искажений на краях
- Увеличение `polyorder` → лучше сохранение высоких частот, но может усиливать шум

## 4. Медианный фильтр

```python
y = signal.medfilt(x, kernel_size=11)          # kernel_size – нечётное
y = scipy.ndimage.median_filter(x, size=11, mode='nearest')   # другая обработка краёв
```

- Заменяет отсчёт медианой значений в окне длины $K$.
- **Нелинейный**: не выполняется принцип суперпозиции, поэтому нет импульсной характеристики и АЧХ (импульсная характеристика формально равна нулю).
- Полностью удаляет одиночные выбросы и короткие импульсы шириной меньше $(K+1)/2$ отсчётов.
- Сохраняет резкие перепады (фронты): медиана на фронте равна значению с той стороны, где больше отсчётов окна.
- Родственные **порядковые фильтры**: максимум, минимум, $k$-я порядковая статистика – `signal.order_filter(x, np.ones(K), rank)`.

## 5. Обратить внимание

| Ошибка | Исправление |
|--------|-------------|
| Знак коэффициентов `a` для БИХ | `a = [1, -(1-alpha)]`, а не `[1, (1-alpha)]` |
| Нормировка коэффициентов | Сумма `b` должна быть 1 (для сохранения постоянной составляющей) |
| Длина `window_length` в `savgol_filter` | Должно быть нечётным |
| Величина `polyorder` | `polyorder` < `window_length` |
| Длина окна в `medfilt` | `kernel_size` должно быть нечётным |
| Края в `medfilt` | Сигнал дополняется нулями, поэтому крайние отсчёты искажаются; используйте `scipy.ndimage.median_filter(..., mode='nearest')` |

## 6. Ссылки на документацию

- [SciPy signal processing](https://docs.scipy.org/doc/scipy/reference/signal.html)
- [scipy.signal.freqz](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.freqz.html)
- [scipy.signal.group_delay](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.group_delay.html)
- [scipy.signal.savgol_filter](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.savgol_filter.html)
- [scipy.signal.medfilt](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.medfilt.html)